# JOINs: INNER, LEFT, Múltiplos e Self-Join
---

## Contexto: o relatório de clientes está errado!

O relatório mensal mostrou que a empresa tem 120 clientes, mas o gerente sabe que são 150.  
Você investiga: o código usa `JOIN` onde deveria usar `LEFT JOIN`, **excluindo clientes sem pedidos**.  
Esse é um bug silencioso clássico em JOINs e você vai aprender a evitá-lo.

Nesta aula você vai aprender:
- A diferença prática entre INNER JOIN e LEFT OUTER JOIN
- JOINs múltiplos encadeados
- Self-join: comparar a tabela com ela mesma
- JOIN + GROUP BY para métricas
- Como inspecionar o SQL gerado para diagnosticar erros

---

## 1. Configuração

In [10]:
# Bibliotecas necessárias
from pathlib import Path
from datetime import datetime, timedelta
from decimal import Decimal
from sqlalchemy import (
    create_engine, select, and_, or_, func, inspect, 
    String, DateTime, Numeric, Integer, Boolean, ForeignKey, CheckConstraint, Index,
    union, union_all, exists, any_
)
from sqlalchemy.orm import (
    DeclarativeBase, Mapped, mapped_column, relationship, Session,
    joinedload, selectinload, aliased
)
from models import Base, Cliente, Produto, ItemPedido, Pedido

# Cria o diretório do banco se não existir
Path("../database").mkdir(exist_ok=True)

# Cria a engine para conectar ao banco SQLite
engine = create_engine("sqlite:///../database/datavendas.db")
# Confirmação visual de que a conexão/engine foi configurada com sucesso
print("Conexão OK ✅")

Conexão OK ✅


In [2]:
# Verificar tabelas com SQLAlchemy
insp = inspect(engine)

print(insp.get_table_names())

['tb_clientes', 'tb_itens_pedidos', 'tb_pedidos', 'tb_produtos']


## 2. JOIN via relacionamento vs JOIN explícito

O SQLAlchemy oferece duas formas de escrever um JOIN:

```python
# Via relacionamento ORM — o SQLAlchemy sabe o ON automaticamente
select(Cliente).join(Cliente.pedidos)

# Explícito — você escreve o ON manualmente
select(Cliente).join(Pedido, Cliente.id == Pedido.cliente_id)
```

**Use relacionamento** quando ele existe e o JOIN é direto.  
**Use explícito** quando precisar de condições extras no ON.

## 3. INNER JOIN — só quem tem par nos dois lados

O `INNER JOIN` funciona como a interseção de dois conjuntos. Ele percorre a tabela de `Clientes` e a tabela de `Pedidos` e só traz os resultados onde existe uma correspondência exata entre as duas, baseada na chave estrangeira.

In [3]:
with Session(engine) as session:

    # INNER JOIN padrão: retorna apenas clientes COM pedidos
    stmt = (
        select(
            Cliente.nome,
            Cliente.estado,
            Pedido.id.label("pedido_id"),
            Pedido.valor_total,
            Pedido.status,
        )
        .join(Cliente.pedidos)  # INNER JOIN via relacionamento
        .order_by(Pedido.valor_total.desc())
    )

    linhas = session.execute(stmt).all()
    total_clientes = session.scalar(select(func.count()).select_from(Cliente))

    print(f"Total de clientes no banco:    {total_clientes}")
    print(f"Linhas no INNER JOIN:          {len(linhas)}")
    print(f"Clientes sem pedido não aparecem!")   
    print("-" * 70)
    # Imprimindo as linhas
    for l in linhas:
        print(f"  {l.nome} | {l.estado} | Pedido #{l.pedido_id} | R${l.valor_total}")

Total de clientes no banco:    21
Linhas no INNER JOIN:          101
Clientes sem pedido não aparecem!
----------------------------------------------------------------------
  Maria Sophia Guerra | RJ | Pedido #46 | R$137396.68
  Dr. Bryan Carvalho | SP | Pedido #13 | R$112483.30
  Laís Santos | SP | Pedido #3 | R$110713.96
  Dr. Diogo Sousa | PA | Pedido #59 | R$103945.50
  Mathias Casa Grande | BA | Pedido #38 | R$97986.41
  Théo Cirino | CE | Pedido #10 | R$96418.85
  Davi Luiz Pimenta | CE | Pedido #32 | R$93806.23
  Ana Liz Monteiro | RJ | Pedido #28 | R$90590.63
  Ana Liz Monteiro | RJ | Pedido #54 | R$90042.40
  Srta. Julia Cassiano | SP | Pedido #91 | R$88708.40
  Alícia Alves | RJ | Pedido #50 | R$86942.48
  Sr. Mathias Guerra | RJ | Pedido #18 | R$84070.02
  Théo Cirino | CE | Pedido #7 | R$78330.55
  Dr. Diogo Sousa | PA | Pedido #17 | R$77475.05
  Joaquim Oliveira | MA | Pedido #31 | R$75017.95
  Mathias Casa Grande | BA | Pedido #20 | R$74795.50
  Maria Sophia Guerra | RJ 

## 4. LEFT OUTER JOIN — todos da esquerda, com ou sem par

`.outerjoin()` retorna **todos os clientes**, mesmo os sem pedido.  
Onde não há par, as colunas do lado direito aparecem como `None`.

In [4]:
with Session(engine) as session:  # Inicia uma sessão de banco de dados para executar consultas

    stmt = (  
        select(  # Seleciona colunas específicas das tabelas
            Cliente.nome,  # Nome do cliente (da tabela Cliente)
            Pedido.id.label("pedido_id"),  # ID do pedido, rotulado como "pedido_id" (da tabela Pedido)
            Pedido.valor_total,  # Valor total do pedido (da tabela Pedido)
        )
        .outerjoin(Cliente.pedidos)  # Realiza um LEFT OUTER JOIN entre Cliente e Pedido via relacionamento 'pedidos'
        .order_by(Cliente.nome)  # Ordena os resultados pelo nome do cliente em ordem alfabética
    )

    linhas = session.execute(stmt).all()  # Executa a consulta e obtém todas as linhas resultantes como uma lista de objetos
    print(f"Linhas no LEFT JOIN: {len(linhas)} (inclui clientes sem pedido)")  # Imprime o número total de linhas, incluindo clientes sem pedidos

    # Clientes sem pedido aparecem com None  
    sem_pedido = [l for l in linhas if l.pedido_id is None]  # Filtra a lista para obter apenas linhas onde pedido_id é None (clientes sem pedidos)
    print(f"Clientes sem nenhum pedido: {len(sem_pedido)}")  # Imprime o número de clientes que não têm nenhum pedido

Linhas no LEFT JOIN: 101 (inclui clientes sem pedido)
Clientes sem nenhum pedido: 0


## 5. JOINs múltiplos — encadeando tabelas

Em análises de vendas, a cadeia completa é:  
**Cliente → Pedido → ItemPedido → Produto**

In [7]:
# Abre uma sessão de conexão com o banco de dados usando o engine configurado
with Session(engine) as session:

    stmt = (
        select(
            
            Cliente.nome.label("cliente"),          
            Pedido.id.label("pedido"),
            Produto.nome.label("produto"),
            Produto.categoria,
            ItemPedido.quantidade,
            ItemPedido.preco_venda,            
            # Calcula o subtotal (quantidade * preço) e renomeia como "subtotal"
            (ItemPedido.quantidade * ItemPedido.preco_venda).label("subtotal")
        )
        
        # Faz JOIN entre Cliente e Pedido (um cliente pode ter vários pedidos)
        .join(Pedido, Cliente.id == Pedido.cliente_id)        
        # Faz JOIN entre Pedido e ItemPedido (um pedido pode ter vários itens)
        .join(ItemPedido, Pedido.id == ItemPedido.pedido_id)        
        # Faz JOIN entre ItemPedido e Produto (cada item corresponde a um produto)
        .join(Produto, ItemPedido.produto_id == Produto.id)        
        # Ordena o resultado pelo valor do subtotal
        .order_by("subtotal")
    )

    # Executa a consulta e retorna todos os resultados
    linhas = session.execute(stmt).all()
    
    # Exibe a quantidade total de linhas retornadas
    print(f"Itens vendidos (com contexto completo): {len(linhas)}")
    print("-" * 80)

    # Percorre apenas os 3 primeiros resultados
    for l in linhas[:3]:
        
        # Exibe as informações formatadas de cada item
        print(f"  {l.cliente} → Pedido #{l.pedido} | {l.quantidade}x {l.produto} | R${l.subtotal}")

Itens vendidos (com contexto completo): 335
--------------------------------------------------------------------------------
  Dra. Evelyn Correia → Pedido #16 | 1x Illum error | R$167.64
  Pietro Porto → Pedido #90 | 1x Illum error | R$167.64
  Mathias Casa Grande → Pedido #48 | 2x Illum error | R$335.28


## 6. Self-join — comparando a tabela com ela mesma

Às vezes a pergunta envolve comparar linhas da **mesma tabela**.  
Exemplo: *"Quais pares de clientes moram na mesma cidade?"*

Para isso, usamos `aliased()` — como ter duas cópias da mesma tabela com nomes diferentes.

In [11]:
# Cria dois aliases (apelidos) para a tabela Cliente
# Isso permite usar a mesma tabela duas vezes na mesma consulta (self join)
ClienteA = aliased(Cliente, name="a")
ClienteB = aliased(Cliente, name="b")

# Monta a consulta
stmt = (
    select(
        # Seleciona o nome do primeiro cliente e renomeia como "cliente_1"
        ClienteA.nome.label("cliente_1"),
        
        # Seleciona o nome do segundo cliente e renomeia como "cliente_2"
        ClienteB.nome.label("cliente_2"),
        ClienteA.cidade,
    )
    
    # Faz um JOIN da tabela Cliente com ela mesma (self join)
    # A condição é clientes que estão na mesma cidade
    .join(ClienteB, ClienteA.cidade == ClienteB.cidade)
    
    # Evita duplicatas como (A,B) e (B,A)
    # Também evita comparar o cliente com ele mesmo
    .where(ClienteA.id < ClienteB.id)
    
    # Ordena o resultado pela cidade
    .order_by(ClienteA.cidade)
)

# Abre uma sessão com o banco de dados
with Session(engine) as session:
    
    # Executa a consulta e retorna todos os resultados
    pares = session.execute(stmt).all()
    
    # Exibe a quantidade de pares encontrados
    print(f"Pares de clientes na mesma cidade: {len(pares)}")
    print("-" * 80)
    
    # Mostra apenas os 5 primeiros pares
    for p in pares[:5]:
        
        # Exibe os clientes que estão na mesma cidade
        print(f"  {p.cliente_1} + {p.cliente_2} — {p.cidade}")

Pares de clientes na mesma cidade: 6
--------------------------------------------------------------------------------
  Melissa Campos + Dra. Evelyn Correia — Belo Horizonte
  Stella Guerra + Anna Liz Sá — Brasília
  Alícia Alves + Ana Liz Monteiro — Duque de Caxias
  Théo Cirino + Davi Luiz Pimenta — Fortaleza
  Sr. Mathias Guerra + Yasmin Fonseca — São Gonçalo


## 7. JOIN + GROUP BY — calculando métricas por grupo

In [12]:
# Receita total por cliente (apenas quem tem pedidos)

# Abre uma sessão com o banco de dados
with Session(engine) as session:

    # Monta a consulta SQL
    stmt = (
        select(
         
            Cliente.nome,         
            Cliente.estado,            
            # Conta quantos pedidos cada cliente fez
            func.count(Pedido.id).label("qtd_pedidos"),            
            # Soma o valor total dos pedidos (receita total por cliente)
            func.sum(Pedido.valor_total).label("receita_total"),            
            # Calcula o ticket médio (média do valor dos pedidos)
            func.avg(Pedido.valor_total).label("ticket_medio"),
        )
        
        # Faz JOIN entre Cliente e Pedido
        # Apenas clientes com pedidos aparecerão no resultado
        .join(Pedido, Cliente.id == Pedido.cliente_id)        
        # Agrupa os resultados por cliente
        # Necessário para usar funções de agregação (count, sum, avg)
        .group_by(Cliente.id)        
        # Ordena pela maior receita total (ordem decrescente)
        .order_by(func.sum(Pedido.valor_total).desc())
    )

    # Executa a consulta e retorna todos os resultados
    resultados = session.execute(stmt).all()

    # Imprime o cabeçalho da tabela formatada
    print(f"{'Cliente':<25} {'UF'} {'Pedidos':>8} {'Receita':>12} {'Ticket Médio':>13}")
    
    # Imprime uma linha separadora
    print("-" * 65)
    
    # Percorre apenas os 8 primeiros resultados
    for r in resultados[:8]:
        
        # Exibe os dados formatados
        print(
            f"{r.nome:<25} "
            f"{r.estado}  "
            f"{r.qtd_pedidos:>8}  "
            f"R${float(r.receita_total or 0):>9.2f}  "
            f"R${float(r.ticket_medio or 0):>9.2f}"
        )

Cliente                   UF  Pedidos      Receita  Ticket Médio
-----------------------------------------------------------------
Maria Sophia Guerra       RJ        10  R$422479.56  R$ 42247.96
Théo Cirino               CE         9  R$412942.75  R$ 45882.53
Stella Guerra             DF        11  R$359972.12  R$ 32724.74
Luiz Miguel Fogaça        GO         8  R$333001.78  R$ 41625.22
Mathias Casa Grande       BA         6  R$329982.24  R$ 54997.04
Ana Liz Monteiro          RJ         5  R$290720.27  R$ 58144.05
Dra. Evelyn Correia       MG         5  R$288066.87  R$ 57613.37
Alícia Alves              RJ         5  R$251422.56  R$ 50284.51


# 8. Visualizando os SQL gerados pelo ORM

Uma das melhores formas de aprender SQLAlchemy é ver o SQL real que ele está enviando para o banco de dados.

In [13]:
from sqlalchemy.dialects import sqlite

# Compilando o comando especificamente para o dialeto SQLite
debug_sql = stmt.compile(dialect=sqlite.dialect(), compile_kwargs={"literal_binds": True})

print("--- SQL COM VALORES REAIS ---")
print(debug_sql)

--- SQL COM VALORES REAIS ---
SELECT tb_clientes.nome, tb_clientes.estado, count(tb_pedidos.id) AS qtd_pedidos, sum(tb_pedidos.valor_total) AS receita_total, avg(tb_pedidos.valor_total) AS ticket_medio 
FROM tb_clientes JOIN tb_pedidos ON tb_clientes.id = tb_pedidos.cliente_id GROUP BY tb_clientes.id ORDER BY sum(tb_pedidos.valor_total) DESC


---

## Exercício: Usando IA para isso

**Prompt para otimizar JOINs múltiplos:**
```
Tenho esta query SQLAlchemy com 3 JOINs encadeados que está lenta.
Sugira: índices que deveriam existir, reescrita mais eficiente,
ou se subconsulta seria mais adequada em alguma parte.
```

```python 
with Session(engine) as session:

    stmt = (
        select(            
            Cliente.nome.label("cliente"),          
            Pedido.id.label("pedido"),
            Produto.nome.label("produto"),
            Produto.categoria,
            ItemPedido.quantidade,
            ItemPedido.preco_venda,            
            (ItemPedido.quantidade * ItemPedido.preco_venda).label("subtotal")
        )
        
        .join(Pedido, Cliente.id == Pedido.cliente_id)        
        .join(ItemPedido, Pedido.id == ItemPedido.pedido_id)        
        .join(Produto, ItemPedido.produto_id == Produto.id)        
        .order_by("subtotal")
    )

    linhas = session.execute(stmt).all()    
    print(f"Itens vendidos (com contexto completo): {len(linhas)}")
    print("-" * 80)

    # Percorre apenas os 3 primeiros resultados
    for l in linhas[:3]:
        
        # Exibe as informações formatadas de cada item
        print(f"  {l.cliente} → Pedido #{l.pedido} | {l.quantidade}x {l.produto} | R${l.subtotal}")
```


---

### Resposta:Código gerado pelo Copilot

In [15]:
with Session(engine) as session:

    subtotal = (ItemPedido.quantidade * ItemPedido.preco_venda)

    stmt = (
        select(            
            Cliente.nome.label("cliente"),          
            Pedido.id.label("pedido"),
            Produto.nome.label("produto"),
            Produto.categoria,
            ItemPedido.quantidade,
            ItemPedido.preco_venda,            
            subtotal.label("subtotal")
        )
        .join(Pedido, Cliente.id == Pedido.cliente_id)
        .join(ItemPedido, Pedido.id == ItemPedido.pedido_id)
        .join(Produto, ItemPedido.produto_id == Produto.id)
        .order_by(subtotal.desc())
        .limit(3)
    )

    linhas = session.execute(stmt).all()
    print(f"Itens vendidos (com contexto completo): {len(linhas)}")
    print("-" * 80)

    # Percorre apenas os 3 primeiros resultados
    for l in linhas[:3]:
        
        # Exibe as informações formatadas de cada item
        print(f"  {l.cliente} → Pedido #{l.pedido} | {l.quantidade}x {l.produto} | R${l.subtotal}")

Itens vendidos (com contexto completo): 3
--------------------------------------------------------------------------------
  Théo Cirino → Pedido #10 | 10x Mollitia ipsam | R$48472.70
  Alícia Alves → Pedido #50 | 10x Accusantium illum nobis | R$47570.60
  Dra. Evelyn Correia → Pedido #16 | 10x Ipsum nihil repudiandae expedita | R$47542.40
